Step 3 Data Visualization

In [3]:
import geopandas as gpd
import pandas as pd
import folium
import branca.colormap as cm

In [ ]:
def render_final_map(geojson_file, csv_file):
    """
    Reads the output files and renders the final map, highlighting the 
    Balanced Route as the primary recommendation.
    """
    print(f"Loading Map Data from {geojson_file}...")
    gdf = gpd.read_file(geojson_file)
    
    print(f"Loading Summary Data from {csv_file}...")
    summary_df = pd.read_csv(csv_file)
    

    # Determine the map center
    first_geom = gdf.iloc[0].geometry
    start_point = first_geom.coords[0] 
    map_center = [start_point[1], start_point[0]] 
    
    # Initialize the base map (Dark mode highlights glowing lines best)
    m = folium.Map(location=map_center, zoom_start=14, tiles="cartodbdark_matter")
    
    # Risk color scale (compatible with different field names)
    risk_col = 'accident_count_current' if 'accident_count_current' in gdf.columns else 'accident_count_50m'
    
    if risk_col in gdf.columns:
        max_risk = max(gdf[risk_col].max(), 5)
        risk_cmap = cm.LinearColormap(
            colors=['#00FF00', '#ADFF2F', '#FFFF00', '#FFA500', '#FF0000'], 
            vmin=0, vmax=max_risk, caption="Risk Level (Accidents)"
        )
        m.add_child(risk_cmap)
        get_color = lambda x: risk_cmap(x)
    else:
        # Default to neon green if no risk data is found
        get_color = lambda x: "#39FF14" 

    # Iterate through all routes and apply different [Line Style Strategies]
    for idx, row in gdf.iterrows():
        # Get route name and convert to lowercase for easier keyword matching
        route_type = str(row.get('route_type', '')).lower()
        dash_style = "10, 10"
        line_weight = 5        
        line_opacity = 0.7     

        if "balanced" in route_type:
            dash_style = None  
            line_weight = 9 
            line_opacity = 1.0 
            
        # Dynamically generate Tooltip hover boxes
        # Display all properties except geometry
        tooltip_html = "<br>".join([f"<b>{k}:</b> {v}" for k, v in row.drop('geometry').items()])
            
        # Draw this line on the map
        folium.GeoJson(
            row.geometry,
            name=row.get('route_type', f'Route {idx}'),
            style_function=lambda f, r_val=row.get(risk_col, 0), d=dash_style, w=line_weight, o=line_opacity: {
                "color": get_color(r_val),
                "weight": w,
                "opacity": o,
                "dashArray": d
            },
            tooltip=folium.Tooltip(tooltip_html)
        ).add_to(m)

    # Add layer control and save
    folium.LayerControl().add_to(m)
    m.save("recommended_route_map.html")
    print("Successfully generated map: recommended_route_map.html")
    
    return m

In [ ]:

# Execute code
if __name__ == "__main__":
    render_final_map("step2_current_routes.geojson", "step2_route_summary.csv")

Loading Map Data from step2_current_routes.geojson...
Loading Summary Data from step2_route_summary.csv...
Successfully generated map: recommended_route_map.html
